## **Registering Codex onto the MAR21 Visium H&E: VALIS vs SimpleITK vs STalign**

Follow-on from `Christian_Causer_et_al_2023.ipynb`, which places the CODEX section on piece 2 of the MAR21 H&E by a rigid outline fit (204°, no flip).
This notebook tests whether published registration tools improve that placement.

Every method is scored with the two checks from the main notebook:
- **AUC carcinoma vs lymphocyte:** for Visium spots the pathologist called invasive carcinoma or lymphocytes, how well the PanCK+ fraction of CODEX cells within 27.5 µm separates them. Independent of the fit, so it's the main test
- **ρ CODEX cells vs haematoxylin:** cell count per spot vs H&E nuclear stain. Partly circular for methods that fit nuclear images (they optimise something close to it)

Kept in a separate notebook so it doesn't clash with edits to the main one. The first cell rebuilds that notebook's state by running its code cells (takes several minutes)

In [ ]:
%%capture
# Rebuild the main notebook's state (data, piece masks, outline search `results`, image fit `fit`, ...)
# by running its code cells here (%run on a notebook would need nbformat, not in the env)
import json as _json

for _c in _json.load(open("Christian_Causer_et_al_2023.ipynb"))["cells"]:
    if _c["cell_type"] == "code":
        _r = get_ipython().run_cell("".join(_c["source"]), silent=True)
        if not _r.success:
            raise RuntimeError(f"main notebook cell {_c['id']} failed") from (
                _r.error_in_exec or _r.error_before_exec
            )

In [ ]:
import json
import os
import subprocess
from glob import glob

# Helper envs for VALIS and STalign (see the docstrings in scripts/); override paths with
# VALIS_PYTHON / STALIGN_PYTHON
conda_root = (
    Path(os.environ["CONDA_EXE"]).parents[1]
    if "CONDA_EXE" in os.environ
    else Path("/opt/anaconda3")
)

### VALIS comparison: feature-based registration of Codex DAPI onto H&E

[VALIS](https://github.com/MathOnco/valis) (Gatenbee et al. 2023) registers serial sections across stains by matching image features, then optionally refines non-rigidly.
Two runs, both fixed = piece-2 H&E crop (haematoxylin), moving = Codex DAPI:
- **From scratch:** raw DAPI, VALIS has to find rotation, flip and scale itself
- **From 204°:** DAPI already resampled onto the H&E with the outline fit (`s0`), VALIS only refines

VALIS needs libvips/pyvips/torch, so it runs in its own conda env through `scripts/valis_codex_to_he.py` (setup in that file's docstring), not in the `virtual-tissue` env

In [ ]:
# Write VALIS inputs to outputs/ (gitignored)
WORK = Path("../../outputs/valis_mar21")
WORK.mkdir(parents=True, exist_ok=True)

he_crop = he[Y0:Y1, X0:X1].copy()
he_crop[~fixed_mask] = 255
# Blank the other tissue pieces so VALIS only sees piece 2
Image.fromarray(he_crop).save(WORK / "he.png")

# (a) From scratch: raw DAPI (level 3) + cell centroids in its pixels
tifffile.imwrite(WORK / "dapi.tif", img[0])
pd.DataFrame(codex.obsm["spatial"] / um_per_codex_px, columns=["x", "y"]).to_csv(
    WORK / "cells_px.csv", index=False
)

# (b) From 204°: DAPI resampled into the H&E crop with the outline fit, cells moved the same way
dapi_init = (warp_codex(img[0], 204, s0) * 255).astype(np.uint8)
# warp() rescales uint8 to 0-1
tifffile.imwrite(WORK / "dapi_init204.tif", dapi_init)
cells_init = (rot_flip(codex.obsm["spatial"], 204, False) + s0 * um_per_hires) / um_per_hires - [
    X0,
    Y0,
]
pd.DataFrame(cells_init, columns=["x", "y"]).to_csv(WORK / "cells_init204_px.csv", index=False)

In [ ]:
# Run VALIS in its own env (~2 min per run); skipped if outputs already exist
# Path to the valis env's python; override with the VALIS_PYTHON env var
VALIS_PY = os.environ.get("VALIS_PYTHON", str(conda_root / "envs/valis/bin/python"))
SCRIPT = Path("../../scripts/valis_codex_to_he.py")

runs = {
    "": [],
    # from scratch, includes the mirrored check
    "_init204": [
        "--moving",
        "dapi_init204.tif",
        "--cells",
        "cells_init204_px.csv",
        "--no-reflections",
    ],
}
for tag, extra in runs.items():
    if (WORK / f"warped_cells{tag}.csv").exists():
        continue
    log = subprocess.run(
        [VALIS_PY, "-W", "ignore", str(SCRIPT), "--work", str(WORK), "--tag", tag, *extra],
        capture_output=True,
        text=True,
    )
    (WORK / f"run{tag}.log").write_text(log.stdout + log.stderr)
    log.check_returncode()

for tag in runs:
    print(tag or "_scratch", json.loads((WORK / f"summary{tag}.json").read_text())["rigid_fit"])
# Rotation/scale/reflection of VALIS's rigid map (scale in H&E px per moving px;
# from scratch, metadata predicts um_per_codex_px / um_per_hires ≈ 0.92)

In [ ]:
# Score every placement with the same checks as above (pathologist AUC, haematoxylin ρ)
def score_xy(xy_um):
    hits = cKDTree(xy_um).query_ball_point(spots_p2, r=27.5)
    n = np.array([len(h) for h in hits])
    frac = np.array([pan_pos[h].mean() if len(h) else np.nan for h in hits])
    car = (n >= 3) & (lab == "Invasive carcinoma")
    lym = (n >= 3) & (lab == "Lymphocytes")
    auc = (
        roc_auc_score(np.r_[np.ones(car.sum()), np.zeros(lym.sum())], np.r_[frac[car], frac[lym]])
        if min(car.sum(), lym.sum()) >= 10
        else np.nan
    )
    return {
        "AUC carcinoma vs lymphocyte": auc,
        "ρ CODEX cells vs haematoxylin": spearmanr(n, hema_spot).statistic,
        "spots with ≥3 cells": int((n >= 3).sum()),
    }


def from_crop(path, cols):
    # H&E-crop px → Visium µm (same frame as spots_p2)
    return (pd.read_csv(path)[cols].to_numpy() + [X0, Y0]) * um_per_hires


xy0 = codex.obsm["spatial"]
placements = {
    "outline fit 204°": rot_flip(xy0, 204, False) + s0 * um_per_hires,
    f"image fit (NCC) {fit['angle']}°": rot_flip(xy0, fit["angle"], False)
    + fit["shift"] * um_per_hires,
    "VALIS rigid, from scratch": from_crop(WORK / "warped_cells.csv", ["x_rigid", "y_rigid"]),
    "VALIS non-rigid, from scratch": from_crop(
        WORK / "warped_cells.csv", ["x_nonrigid", "y_nonrigid"]
    ),
    "VALIS rigid, from 204°": from_crop(WORK / "warped_cells_init204.csv", ["x_rigid", "y_rigid"]),
    "VALIS non-rigid, from 204°": from_crop(
        WORK / "warped_cells_init204.csv", ["x_nonrigid", "y_nonrigid"]
    ),
}
pd.DataFrame({k: score_xy(v) for k, v in placements.items()}).T.round(3)

In [ ]:
# VALIS's own overlap thumbnails (before / rigid / non-rigid)
fig, ax = plt.subplots(2, 3, figsize=(18, 10))
for row, tag in zip(ax, ["", "_init204"]):
    for a, step in zip(row, ["original", "rigid", "non_rigid"]):
        a.imshow(
            Image.open(
                WORK / f"valis{tag}/codex_to_he{tag}/overlaps/codex_to_he{tag}_{step}_overlap.png"
            )
        )
        a.set_title(f"{tag.strip('_') or 'from scratch'}: {step}")
        a.axis("off")
plt.tight_layout()

#### VALIS result

Measured 2026-11-20 (valis-wsi 1.2.0); VALIS gives identical results on rerun

| placement | AUC | ρ haematoxylin | spots ≥3 cells |
|---|---|---|---|
| outline fit 204° | 0.840 | 0.214 | 773 |
| VALIS rigid, from scratch | 0.432 | 0.003 | 477 |
| VALIS non-rigid, from scratch | 0.348 | 0.022 | 481 |
| VALIS rigid, from 204° | 0.535 | 0.167 | 712 |
| VALIS non-rigid, from 204° | 0.765 | 0.293 | 777 |

- **From scratch, VALIS fails:** it settles on 315°, no flip, scale 0.735 (the pixel sizes predict 0.92); AUC is below chance
- **From 204°, its rigid step makes the fit worse:** another −5° and 0.85× scale
- Only 4–6 feature matches between DAPI and haematoxylin at ~3.3 µm/px, too few for a reliable transform
- Non-rigid from 204° raises ρ but lowers AUC (0.84 → 0.77); ρ is partly circular, AUC is the fairer test
- `--no-rigid` (non-rigid only) moved no cells at all, so VALIS seems to skip its non-rigid step without a rigid one
- Possible fix not tried: a higher-resolution H&E (only the hires PNG is in the upload)

### SimpleITK affine + B-spline (the Causer et al. method)

The paper's PhenoCycler–Visium registration: SimpleITK, CODEX DAPI as moving image, greyscale H&E as fixed, centre the images, affine, then B-spline, with Mattes mutual information as the metric for both steps.
They set the first rotation by hand. Here we start from the 204° outline fit instead (the moving image is `dapi_init`, written in the VALIS section). Runs take a few minutes: single-threaded, all pixels, for reproducibility.

Differences from the paper:
- fixed image is tested as both greyscale (paper) and haematoxylin (this notebook's choice)
- B-spline control grid tested at 4×4 and 8×8 (crop ≈ 4.2 × 2.7 mm, so roughly 1 mm and 500 µm control-point spacing)
- the moving tissue mask is not used; the fixed mask is piece 2 only

In [ ]:
import SimpleITK as sitk

# Deterministic: one thread and every pixel in the metric. With random sampling and/or
# several threads, repeat runs gave B-spline 8×8 AUC anywhere in 0.925–0.976
sitk.ProcessObject_SetGlobalDefaultNumberOfThreads(1)

# One sitk pixel = one hires H&E px (spacing 1); all images are in the piece-2 crop frame


def to_sitk(a):
    a = a.astype(np.float32)
    return sitk.GetImageFromArray((a - a.min()) / (np.ptp(a) + 1e-9))


sitk_fixed = {
    "haematoxylin": to_sitk(ndi.gaussian_filter(rgb2hed(he[Y0:Y1, X0:X1])[..., 0], 1)),
    # Paper: greyscale; inverted so tissue is bright like DAPI
    "greyscale (paper)": to_sitk(1 - he[Y0:Y1, X0:X1].mean(axis=2) / 255),
}
sitk_fmask = sitk.GetImageFromArray(fixed_mask.astype(np.uint8))


def mi_method(fmask=sitk_fmask):
    R = sitk.ImageRegistrationMethod()
    R.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    R.SetMetricSamplingStrategy(R.NONE)  # all pixels
    R.SetMetricFixedMask(fmask)
    R.SetInterpolator(sitk.sitkLinear)
    # Coarse-to-fine: 4x, 2x, 1x downsampling
    R.SetShrinkFactorsPerLevel([4, 2, 1])
    R.SetSmoothingSigmasPerLevel([2, 1, 0])
    R.SmoothingSigmasAreSpecifiedInPhysicalUnitsOff()
    return R


def register_affine(F, M):
    R = mi_method()
    R.SetOptimizerAsRegularStepGradientDescent(
        learningRate=1.0, minStep=1e-4, numberOfIterations=300
    )
    R.SetOptimizerScalesFromPhysicalShift()
    init = sitk.CenteredTransformInitializer(
        F, M, sitk.AffineTransform(2), sitk.CenteredTransformInitializerFilter.GEOMETRY
    )
    R.SetInitialTransform(init, inPlace=False)
    tx = R.Execute(F, M)
    if tx.GetName() == "CompositeTransform":
        tx = tx.GetNthTransform(0)
    return sitk.AffineTransform(tx)


def register_bspline(F, M, affine, mesh):
    R = mi_method()
    R.SetOptimizerAsLBFGSB(
        gradientConvergenceTolerance=1e-5,
        numberOfIterations=100,
        maximumNumberOfCorrections=5,
        maximumNumberOfFunctionEvaluations=1000,
    )
    bs = sitk.BSplineTransformInitializer(F, [mesh, mesh], order=3)
    R.SetMovingInitialTransform(affine)
    R.SetInitialTransform(bs, inPlace=True)
    R.Execute(F, M)
    # Composite applies the last-added first: B-spline, then affine
    return sitk.CompositeTransform([affine, bs])


def moving_to_fixed(tx, pts, F):
    # sitk transforms map fixed → moving; CODEX cells are in moving space, so invert
    if tx.GetName() == "AffineTransform":
        inv = tx.GetInverse()
    else:
        field = sitk.TransformToDisplacementField(
            tx, sitk.sitkVectorFloat64, F.GetSize(), F.GetOrigin(), F.GetSpacing(), F.GetDirection()
        )
        inv = sitk.DisplacementFieldTransform(
            sitk.InvertDisplacementField(
                field, maximumNumberOfIterations=50, enforceBoundaryCondition=False
            )
        )
    return np.array([inv.TransformPoint(p) for p in pts.tolist()])


def folding(tx, F):
    # Jacobian determinant ≤ 0 means the warp folds tissue over itself
    field = sitk.TransformToDisplacementField(
        tx, sitk.sitkVectorFloat64, F.GetSize(), F.GetOrigin(), F.GetSpacing(), F.GetDirection()
    )
    jac = sitk.GetArrayFromImage(sitk.DisplacementFieldJacobianDeterminant(field))[fixed_mask]
    return jac.min(), (jac <= 0).mean()

In [ ]:
# Affine → B-spline from the 204° start, for both fixed images and two grid sizes (~1 min)
sitk_moving = to_sitk(dapi_init)
crop_to_um = lambda p: (p + [X0, Y0]) * um_per_hires

sitk_rows = {"start: 204° outline": score_xy(crop_to_um(cells_init))}
sitk_tx, sitk_pts = {}, {}
for fname, F in sitk_fixed.items():
    aff = register_affine(F, sitk_moving)
    sitk_tx[f"{fname}: affine"] = aff
    for mesh in [4, 8]:
        sitk_tx[f"{fname}: affine + B-spline {mesh}×{mesh}"] = register_bspline(
            F, sitk_moving, aff, mesh
        )
for name, tx in sitk_tx.items():
    F = sitk_fixed[name.split(":")[0]]
    sitk_pts[name] = moving_to_fixed(tx, cells_init, F)
    move = np.median(np.linalg.norm(sitk_pts[name] - cells_init, axis=1)) * um_per_hires
    sitk_rows[name] = {
        **score_xy(crop_to_um(sitk_pts[name])),
        "median cell move (µm)": move,
        "min Jacobian": folding(tx, F)[0] if "B-spline" in name else np.nan,
    }
pd.DataFrame(sitk_rows).T.round(3)

In [ ]:
# Control: run the same affine → B-spline (haematoxylin, 8×8) from WRONG starting placements.
# If it reaches high AUC from anywhere, the gain is just free-form warping, not alignment


def init_from(angle, flip):
    row = results[(results.piece == 2) & (results.angle == angle) & (results.flip == flip)].iloc[0]
    shift = np.asarray(row["shift"], float)
    a = np.deg2rad(angle)
    R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
    mirror = np.array([-1, 1]) if flip else np.array([1, 1])

    def inv(xy):  # crop px → CODEX level-3 px (undo rotate, then undo mirror)
        return (((xy + [X0, Y0] - shift) * um_per_hires) @ R) * mirror / um_per_codex_px

    im = warp(img[0], inv, output_shape=fixed.shape, order=1, cval=0)
    cells = (rot_flip(codex.obsm["spatial"], angle, flip) + shift * um_per_hires) / um_per_hires - [
        X0,
        Y0,
    ]
    return im, cells


F = sitk_fixed["haematoxylin"]
control = {}
for angle, flip in [(204, False), (234, False), (174, False), (345, True), (102, False)]:
    im, cells = init_from(angle, flip)
    M = to_sitk(im)
    aff = register_affine(F, M)
    bs = register_bspline(F, M, aff, 8)
    jmin, jfold = folding(bs, F)
    control[f"{angle}°, flip={flip}"] = {
        "AUC start": score_xy(crop_to_um(cells))["AUC carcinoma vs lymphocyte"],
        "AUC affine": score_xy(crop_to_um(moving_to_fixed(aff, cells, F)))[
            "AUC carcinoma vs lymphocyte"
        ],
        "AUC B-spline": score_xy(crop_to_um(moving_to_fixed(bs, cells, F)))[
            "AUC carcinoma vs lymphocyte"
        ],
        "min Jacobian": jmin,
        "% folded": 100 * jfold,
    }
pd.DataFrame(control).T.round(3)

In [ ]:
# Paired bootstrap over piece-2 spots: is the B-spline AUC gain at 204° more than noise?
def panck_frac(xy_um):
    hits = cKDTree(xy_um).query_ball_point(spots_p2, r=27.5)
    return np.array([len(h) for h in hits]), np.array(
        [pan_pos[h].mean() if len(h) else np.nan for h in hits]
    )


# 4×4: never folded (min Jacobian > 0) in any run; 8×8 folded slightly in one
best = "haematoxylin: affine + B-spline 4×4"
n0, f0 = panck_frac(crop_to_um(cells_init))
n1, f1 = panck_frac(crop_to_um(sitk_pts[best]))
ok = (n0 >= 3) & (n1 >= 3) & np.isin(lab, ["Invasive carcinoma", "Lymphocytes"])
y, a0, a1 = (lab[ok] == "Invasive carcinoma").astype(int), f0[ok], f1[ok]
idx_c, idx_l = np.where(y == 1)[0], np.where(y == 0)[0]
rng = np.random.default_rng(0)
diffs = []
for _ in range(2000):
    # Resample carcinoma and lymphocyte spots separately
    b = np.r_[rng.choice(idx_c, len(idx_c)), rng.choice(idx_l, len(idx_l))]
    diffs.append(roc_auc_score(y[b], a1[b]) - roc_auc_score(y[b], a0[b]))
print(
    f"AUC gain {roc_auc_score(y, a1) - roc_auc_score(y, a0):+.3f}, "
    f"95% CI [{np.percentile(diffs, 2.5):+.3f}, {np.percentile(diffs, 97.5):+.3f}] "
    f"(n carcinoma {len(idx_c)}, n lymphocyte {len(idx_l)})"
)

# Keep the B-spline placement for downstream use: CODEX cells in MAR21 Visium µm
codex.obsm["spatial_mar21_um"] = crop_to_um(sitk_pts[best])

In [ ]:
# Overlay: H&E haematoxylin (pink) vs CODEX DAPI (green), white = agreement
F = sitk_fixed["haematoxylin"]
f = sitk.GetArrayFromImage(F)
fig, ax = plt.subplots(1, 3, figsize=(30, 8))
for a, (title, tx) in zip(
    ax,
    [
        ("204° outline start", sitk.Transform(2, sitk.sitkIdentity)),
        ("+ affine", sitk_tx["haematoxylin: affine"]),
        ("+ affine + B-spline 4×4", sitk_tx[best]),
    ],
):
    m = sitk.GetArrayFromImage(sitk.Resample(sitk_moving, F, tx, sitk.sitkLinear, 0.0))
    fn = np.clip(f / np.percentile(f[fixed_mask], 99), 0, 1)
    mn = np.clip(m / np.percentile(m[m > 0], 99), 0, 1)
    a.imshow(np.dstack([fn, mn, fn]))
    a.set_title(title, fontsize=16)
    a.axis("off")
plt.tight_layout()

#### SimpleITK result

Measured 2026-11-20 (SimpleITK 2.5.6, single thread, all pixels: reruns are identical)

| placement (from 204°) | AUC | ρ haematoxylin | median cell move | min Jacobian |
|---|---|---|---|---|
| start: 204° outline | 0.840 | 0.214 | – | – |
| haematoxylin: affine | 0.887 | 0.290 | 73 µm | – |
| haematoxylin: + B-spline 4×4 | **0.944** | 0.299 | 121 µm | 0.57 |
| haematoxylin: + B-spline 8×8 | 0.954 | 0.226 | 138 µm | **−0.08** (folds) |
| greyscale (paper): affine | 0.870 | 0.263 | 58 µm | – |
| greyscale (paper): + B-spline 4×4 | 0.904 | 0.266 | 195 µm | 0.59 |
| greyscale (paper): + B-spline 8×8 | 0.963 | 0.127 | 116 µm | **−0.01** (folds) |

- **B-spline improves the independent check:** 4×4 (haematoxylin) vs the 204° start, AUC gain +0.108, 95% CI [+0.048, +0.172] (paired bootstrap, 189 carcinoma / 53 lymphocyte spots)
- **Robust to the start, measured in a scratch run:** with the start shifted ±1 hires px (~3 µm), 4×4 gave AUC 0.90–0.97 and never folded; 8×8 gave 0.95–0.97 but folded in one of five
- **Not deterministic by default:** with random metric sampling and/or several threads, repeat runs of 8×8 gave AUC 0.925–0.976. Hence one thread and all pixels here
- **Wrong-start control (8×8):** from 234°, 174°, 102° and 345° (mirrored), AUC 0.33–0.52, with 0–26% of the tissue folded. The gain needs a correct start; it isn't just flexible warping. This also counts against the mirrored 345° orientation
- 8×8 folds tissue in some runs, so **4×4 is the recommended setting**. It is saved as `codex.obsm["spatial_mar21_um"]`
- Medians of 70–200 µm of cell movement are larger than a spot, so check the overlay before trusting local detail

### STalign (LDDMM: affine + diffeomorphism)

[STalign](https://github.com/JEFworks-Lab/STalign) (Clifton et al. 2023) fits an affine map plus a smooth, invertible deformation by gradient descent, matching image intensities through a fitted contrast map.
It runs in its own env through `scripts/stalign_codex_to_he.py`, starting from the 204° placement (`dapi_init204.tif`, `cells_init204_px.csv`) with its rotation pre-fit switched off.

Settings that mattered (found by trial on this data):
- STalign's automatic step sizes came out at ~1e-12 (linear) and 2e-5 µm (translation), so nothing moved. Explicit steps (epL 2e-8, epT 0.2, epV 20) are used here; epV 2000 diverged (median cell move 1.1 mm)
- `sigmaM` set to the target's standard deviation; STalign's default (1.0) gives the matching term almost no weight for 0–1 images
- 200 affine-only iterations first, then 1800 with the deformation; images downsampled 2× (~6.6 µm/px)

STalign gives identical results on rerun with identical inputs, but see the instability note below

In [ ]:
# Run STalign from 204° with four source/target settings, plus the wrong-start control (~30 s each)
STALIGN_PY = os.environ.get("STALIGN_PYTHON", str(conda_root / "envs/stalign/bin/python"))
STALIGN_SCRIPT = Path("../../scripts/stalign_codex_to_he.py")
steps = ["--epL", "2e-8", "--epT", "2e-1", "--epV", "2e1"]


def run_stalign(key, extra):
    # Returns warped cells (H&E-crop px) and STalign's summary; cached by key
    hits = glob(str(WORK / f"warped_cells_stalign_*_nb{key}.csv"))
    if not hits:
        log = subprocess.run(
            [
                STALIGN_PY,
                "-W",
                "ignore",
                str(STALIGN_SCRIPT),
                "--work",
                str(WORK),
                "--um-per-px",
                str(um_per_hires),
                *steps,
                *extra,
                "--tag",
                f"_nb{key}",
            ],
            capture_output=True,
            text=True,
        )
        (WORK / f"run_stalign_nb{key}.log").write_text(log.stdout + log.stderr)
        log.check_returncode()
        hits = glob(str(WORK / f"warped_cells_stalign_*_nb{key}.csv"))
    f = Path(hits[0])
    info = json.loads(
        (f.parent / f.name.replace("warped_cells_", "").replace(".csv", ".json")).read_text()
    )
    return pd.read_csv(f)[["x", "y"]].to_numpy(), info


stalign_configs = {
    "DAPI → H&E RGB": ["--source", "dapi", "--target", "rgb"],
    "cell density → H&E RGB": ["--source", "density", "--target", "rgb"],
    "DAPI → haematoxylin": ["--source", "dapi", "--target", "hem"],
    "DAPI → H&E RGB, a = 250 µm": ["--source", "dapi", "--target", "rgb", "--a", "250"],
}
stalign_rows, stalign_pts = {"start: 204° outline": score_xy(crop_to_um(cells_init))}, {}
for i, (name, extra) in enumerate(stalign_configs.items()):
    pts, info = run_stalign(f"c{i}", extra)
    stalign_pts[name] = pts
    stalign_rows[name] = {
        **score_xy(crop_to_um(pts)),
        "median cell move (µm)": info["median_cell_move_um"],
        "affine det": info["affine_det"],
    }
display(pd.DataFrame(stalign_rows).T.round(3))

# Wrong-start control: same pipeline (DAPI → H&E RGB) from wrong placements
stalign_control = {}
for angle, flip in [(204, False), (234, False), (174, False), (345, True), (102, False)]:
    tag = f"{angle}{'f' if flip else ''}"
    if angle == 204:
        pts, cells = stalign_pts["DAPI → H&E RGB"], cells_init
    else:
        im, cells = init_from(angle, flip)
        tifffile.imwrite(WORK / f"dapi_init{tag}.tif", (im * 255).astype(np.uint8))
        pd.DataFrame(cells, columns=["x", "y"]).to_csv(
            WORK / f"cells_init{tag}_px.csv", index=False
        )
        pts, _ = run_stalign(
            f"start{tag}", ["--moving", f"dapi_init{tag}.tif", "--cells", f"cells_init{tag}_px.csv"]
        )
    stalign_control[f"{angle}°, flip={flip}"] = {
        "AUC start": score_xy(crop_to_um(cells))["AUC carcinoma vs lymphocyte"],
        "AUC STalign": score_xy(crop_to_um(pts))["AUC carcinoma vs lymphocyte"],
    }
pd.DataFrame(stalign_control).T.round(3)

#### STalign result

Measured 2026-11-20 (STalign 2.0.0 at commit 8a84bf1)

| from 204° | AUC | ρ haematoxylin | median cell move |
|---|---|---|---|
| start: 204° outline | 0.840 | 0.214 | – |
| DAPI → H&E RGB | 0.476 | −0.029 | 518 µm |
| cell density → H&E RGB | 0.902 | 0.291 | 161 µm |
| DAPI → haematoxylin | 0.848 | 0.290 | 85 µm |
| DAPI → H&E RGB, a = 250 µm | 0.540 | −0.131 | 537 µm |

Wrong-start control (DAPI → H&E RGB): AUC 0.47–0.72; no better than from 204°

- **STalign is unstable here:** changing the pixel size from 3.278 to 3.27795 µm (0.0015%) flips DAPI → H&E RGB between two solutions, a 178 µm median move (AUC 0.90 in a scratch run) and a 518 µm one (AUC 0.48, above)
- This is a property of the optimisation with these hand-set step sizes: its automatic steps don't move at all, and larger ones diverge. So no single STalign result here is dependable, including the cell-density one (AUC 0.90)
- Not tuned further. A coarse-to-fine schedule or landmarks might stabilise it

### Summary

In [ ]:
# All methods side by side
summary = {
    "outline fit 204° (main notebook)": crop_to_um(cells_init),
    f"image fit, NCC rigid {fit['angle']}° (main notebook)": rot_flip(
        codex.obsm["spatial"], fit["angle"], False
    )
    + fit["shift"] * um_per_hires,
    "VALIS non-rigid, from 204°": from_crop(
        WORK / "warped_cells_init204.csv", ["x_nonrigid", "y_nonrigid"]
    ),
    "SimpleITK affine + B-spline 4×4 (haematoxylin)": crop_to_um(sitk_pts[best]),
    "STalign cell density → H&E RGB (unstable, see above)": crop_to_um(
        stalign_pts["cell density → H&E RGB"]
    ),
}
pd.DataFrame({k: score_xy(v) for k, v in summary.items()}).T.round(3)

| method | AUC | ρ haematoxylin | verdict |
|---|---|---|---|
| outline fit 204° (main notebook) | 0.840 | 0.214 | starting point |
| image fit, NCC rigid 209° (main notebook) | 0.933 | 0.270 | **nearly as good as the best, rigid only** |
| VALIS non-rigid, from 204° | 0.765 | 0.293 | worse; VALIS fails from scratch |
| SimpleITK affine + B-spline 4×4 (haematoxylin) | **0.944** | 0.299 | best; robust to the start; no folding |
| STalign cell density → H&E RGB | 0.902 | 0.291 | unstable; not dependable |

- **Use SimpleITK affine + B-spline 4×4** (`codex.obsm["spatial_mar21_um"]`). It's the Causer et al. method, started from the 204° fit instead of a manual rotation
- **Most of the gain over 204° is a ~5° rotation:** the rigid NCC fit at 209° already reaches AUC 0.933, and the B-spline adds about 0.01 on top. The bootstrap above compares against 204°, not 209°
- Next test: start SimpleITK from the 209° NCC fit rather than 204°, and see whether the B-spline adds anything beyond the rotation
- The checks are limited: only 53 lymphocyte spots, and the 0.84–0.97 range is driven by those few